In [2]:
#Per TPM Spec: https://trustedcomputinggroup.org/wp-content/uploads/TCG_Algorithm_Registry_Rev_1.22.pdf
n = 115792089237314936872688561244471742058035595988840268584488757999429535617037
p = 115792089237314936872688561244471742058375878355761205198700409522629664518163

A = 0
B = 3

E = EllipticCurve(GF(p), [A, B])

G = E(1,2)
k = 12

#------- AI ------------------------------
# Fp2
Fp2.<i> = GF(p^2)
# same curve over Fp2
E2 = EllipticCurve(Fp2, [A, B])
# move G1 generator into extension field
G = E2(G)

# find a valid G2 point
cofactor = (p^2 - 1) // n
G2 = cofactor * E2.random_point()
#----------------------------------------

print("------------------------------------------- Set Up------------------------------------------")
print(f"g1_x: {int(G[0])}")
print(f"g1_y: {int(G[1])}")

# Print G2 generator components
print(f"g2_x_a: {list(int(G2[0].list()[0]).to_bytes(32,"big"))}")
print(f"g2_x_b: {list(int(G2[0].list()[1]).to_bytes(32,"big"))}")
print(f"g2_y_a: {list(int(G2[1].list()[0]).to_bytes(32,"big"))}")
print(f"g2_y_b: {list(int(G2[1].list()[1]).to_bytes(32,"big"))}")
print("---------------------------------------------------------------------------------------------")

------------------------------------------- Set Up------------------------------------------
g1_x: 1
g1_y: 2
g2_x_a: [213, 209, 35, 160, 254, 115, 170, 80, 171, 172, 176, 99, 113, 12, 247, 87, 16, 81, 120, 17, 110, 28, 124, 83, 162, 153, 77, 155, 109, 195, 94, 249]
g2_x_b: [224, 121, 39, 169, 212, 153, 202, 254, 239, 60, 61, 90, 223, 66, 32, 239, 156, 187, 52, 167, 149, 24, 253, 23, 203, 60, 179, 12, 41, 198, 140, 10]
g2_y_a: [88, 112, 228, 176, 21, 67, 104, 192, 242, 166, 143, 123, 104, 12, 24, 243, 231, 83, 82, 182, 25, 230, 82, 239, 142, 145, 211, 184, 221, 100, 91, 165]
g2_y_b: [246, 96, 86, 176, 81, 26, 83, 55, 49, 164, 45, 60, 248, 249, 255, 179, 129, 205, 76, 194, 28, 222, 101, 13, 67, 17, 131, 91, 116, 7, 90, 201]
---------------------------------------------------------------------------------------------


In [3]:
sk_x = randint(1, n-1)
sk_y = randint(1, n-1)

X = sk_x * G2
Y = sk_y * G2

print("------------------------------------------Issuer Keys-----------------------------------------")
print(f"sk_x: {int(sk_x)}")
print(f"sk_y: {int(sk_y)}")

print(f"X_a: {list(int(X[0].list()[0]).to_bytes(32,"big"))}")
print(f"X_b: {list(int(X[0].list()[1]).to_bytes(32,"big"))}")
print(f"Y_a: {list(int(Y[1].list()[0]).to_bytes(32,"big"))}")
print(f"Y_b: {list(int(Y[1].list()[1]).to_bytes(32,"big"))}")
print("----------------------------------------------------------------------------------------------")


------------------------------------------Issuer Keys-----------------------------------------
sk_x: 113757801524762723913524620067999213450072541613824098411213053947843580119743
sk_y: 113616277033824222014384903147134222026370730406720199895703812673521178671763
X_a: [169, 50, 85, 56, 22, 143, 45, 208, 128, 92, 92, 192, 41, 128, 200, 223, 44, 2, 57, 57, 127, 151, 200, 167, 13, 178, 67, 173, 209, 38, 225, 170]
X_b: [142, 103, 91, 219, 249, 141, 157, 24, 175, 36, 214, 194, 112, 11, 1, 51, 26, 178, 233, 193, 106, 203, 204, 192, 225, 30, 122, 10, 238, 86, 37, 73]
Y_a: [55, 189, 128, 150, 167, 28, 120, 95, 212, 66, 251, 162, 178, 49, 176, 215, 196, 130, 133, 75, 188, 7, 5, 107, 243, 226, 183, 45, 4, 62, 141, 10]
Y_b: [106, 131, 123, 128, 113, 103, 5, 133, 19, 161, 204, 70, 132, 137, 250, 9, 213, 158, 137, 242, 67, 177, 247, 173, 89, 81, 133, 77, 168, 153, 201, 35]
----------------------------------------------------------------------------------------------


In [4]:
Q_x = [169, 50, 204, 84, 69, 112, 196, 2, 170, 235, 84, 170, 220, 186, 133, 150, 66, 64, 22, 118, 225, 189, 10, 24, 131, 131, 17, 82, 120, 20, 36, 133]
Q_y = [90, 143, 41, 67, 121, 122, 6, 102, 90, 243, 65, 194, 236, 33, 157, 14, 38, 150, 54, 0, 198, 44, 151, 119, 180, 167, 240, 7, 173, 0, 187, 151]

Q_x_int = int.from_bytes(Q_x,"big")
Q_y_int = int.from_bytes(Q_y,"big")
Q = E(Q_x_int, Q_y_int)
Q = E2(Q)

print("-------------------------------------TPM Values-----------------------------------------------")
print(Q_x_int)
print(Q_y_int)
print("----------------------------------------------------------------------------------------------")

-------------------------------------TPM Values-----------------------------------------------
76530623992014141396574573350866305709503765925766633226171737041773767435397
40961100293466914489954188924660758113044736346186255863046168770807217568663
----------------------------------------------------------------------------------------------


In [5]:
r = randint(1, n-1)

A = r * G
B = sk_y * A
C = (sk_x * A) + ((r * sk_x * sk_y) * Q )

print("----------------------------------------Credentials-------------------------------------------")
print(f"A_x: {list(int(A[0]).to_bytes(32,"big"))}")
print(f"A_y: {list(int(A[1]).to_bytes(32,"big"))}")

print(f"B_x: {list(int(B[0]).to_bytes(32,"big"))}")
print(f"B_y: {list(int(B[1]).to_bytes(32,"big"))}")

print(f"C_x: {list(int(C[0]).to_bytes(32,"big"))}")
print(f"C_y: {list(int(C[1]).to_bytes(32,"big"))}")
print("----------------------------------------------------------------------------------------------")


----------------------------------------Credentials-------------------------------------------
A_x: [37, 111, 246, 121, 67, 254, 143, 255, 12, 194, 57, 206, 71, 244, 107, 150, 164, 215, 96, 148, 255, 235, 8, 230, 14, 103, 25, 169, 254, 237, 149, 161]
A_y: [1, 106, 51, 57, 56, 99, 24, 92, 29, 170, 0, 151, 206, 28, 134, 205, 67, 53, 62, 183, 121, 180, 206, 87, 170, 136, 194, 210, 235, 227, 204, 114]
B_x: [108, 88, 13, 54, 170, 85, 182, 77, 95, 82, 182, 189, 173, 62, 8, 246, 242, 164, 241, 169, 146, 33, 135, 58, 216, 174, 30, 243, 68, 125, 247, 153]
B_y: [76, 184, 119, 61, 73, 145, 28, 213, 161, 192, 178, 166, 188, 124, 15, 168, 206, 126, 23, 47, 71, 77, 227, 144, 213, 105, 48, 165, 197, 130, 226, 54]
C_x: [244, 44, 116, 194, 27, 107, 3, 94, 102, 108, 199, 39, 224, 35, 27, 1, 157, 56, 123, 224, 13, 94, 48, 170, 27, 226, 65, 172, 139, 17, 45, 233]
C_y: [100, 184, 176, 82, 104, 73, 135, 132, 128, 9, 86, 216, 156, 120, 135, 216, 218, 207, 196, 79, 22, 235, 252, 242, 194, 225, 222, 248, 47, 2

In [6]:
D_x =[168, 170, 26, 109, 145, 24, 96, 205, 17, 69, 84, 11, 137, 217, 88, 62, 55, 135, 234, 88, 179, 1, 89, 0, 56, 61, 128, 191, 85, 82, 190, 212]
D_y =[150, 105, 228, 142, 92, 85, 92, 74, 44, 47, 191, 206, 22, 197, 118, 7, 72, 26, 94, 250, 41, 121, 63, 239, 239, 186, 185, 225, 217, 4, 77, 141]

D_x_int = int.from_bytes(D_x,"big")
D_y_int = int.from_bytes(D_y,"big")
D = E(D_x_int, D_y_int)
D = E2(D)

print("------------------------------------Verify----------------------------------------------------")
# hˆ(A, Y ) = hˆ(B, P2)
print(A.tate_pairing(Y, n, k,p) == B.tate_pairing(G2, n, k,p))

# hˆ(A + D, X) = hˆ(C, P2)
print((A + D).tate_pairing(X,n,k,p) == C.tate_pairing(G2,n,k,p))
print("----------------------------------------------------------------------------------------------")

------------------------------------Verify----------------------------------------------------
True
True
----------------------------------------------------------------------------------------------
